In [2]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import optuna
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score


DATA_PATH = os.path.join('..', 'vr_legibility_train.csv')

c:\Users\psgve\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1. 데이터 로드 및 전처리
df = pd.read_csv(DATA_PATH)[["yaw", "pitch", "size", "label"]]

# 특성(X)과 타겟(y) 분리
X = df.drop('label', axis=1)
y = df['label']

# 학습/테스트 데이터 분할 (StratifiedKFold 적용)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [4]:
# 2. 파이프라인 (스케일링 + 모델) 구축
# ---------------------------------------------------------
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(class_weight='balanced', random_state=42))
])

In [5]:
# 3. 교차 검증 객체 생성 및 Optuna를 활용한 하이퍼파라미터 튜닝
f1_scores = []
acc_scores = []
best_params_per_fold = []

def objective(trial, X_train, y_train, pipeline):
    kernel = trial.suggest_categorical('svm__kernel', ['rbf', 'linear', 'poly'])
   
    params = {'svm__kernel': kernel}
        
    if kernel == 'rbf':
        params['svm__C'] = trial.suggest_float('svm__C', 0.1, 1000.0, log=True)
        params['svm__gamma'] = trial.suggest_float('svm__gamma', 1e-3, 10.0, log=True)
            
    elif kernel == 'linear':
        params['svm__C'] = trial.suggest_float('svm__C', 1e-3, 10.0, log=True)
            
    elif kernel == 'poly':
        params['svm__C'] = trial.suggest_float('svm__C', 0.1, 1.0, log=True)
        params['svm__gamma'] = trial.suggest_float('svm__gamma', 1e-3, 1.0, log=True)
        params['svm__degree'] = trial.suggest_int('svm__degree', 2, 3)
            
    # 파이프라인에 파라미터 적용
    pipeline.set_params(**params)
        
    scores = cross_val_score(
    pipeline, X_train, y_train, 
    cv=5, scoring='f1', n_jobs=-1
    )
    return scores.mean()

In [7]:
# 4. Outer CV 루프에서 Optuna 최적화 실행
import warnings
warnings.filterwarnings('ignore') # 경고 메시지 무시
optuna.logging.set_verbosity(optuna.logging.WARNING) # 로그가 너무 많이 뜨는 것을 방지 

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
    print(f"\n--- Outer Fold {fold_idx + 1} / 5 ---")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Optuna Study 생성 및 최적화 실행
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda trial: objective(trial, X_train, y_train, pipeline), n_trials=120, n_jobs=1) 

    best_params_inner = study.best_params
    best_score_inner = study.best_value
    
    print("-" * 40)
    print(f"Best Params: {best_params_inner}")
    print(f"Best CV F1:  {best_score_inner:.4f}")

    # 최고 성능 파라미터로 전체 훈련 데이터(X_train) 재학습
    best_model_inner = pipeline.set_params(**best_params_inner)
    best_model_inner.fit(X_train, y_train)

    # 검증 데이터 평가
    y_pred = best_model_inner.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    
    f1_scores.append(f1)
    acc_scores.append(acc)
    best_params_per_fold.append(best_params_inner)

print("\n"+"-" * 10 + " FINAL RESULTS " + "-" * 10)
print(f"최종 5-Fold 평균 F1 Score: {np.mean(f1_scores):.4f} (± {np.std(f1_scores):.4f})")
print(f"최종 5-Fold 평균 Accuracy: {np.mean(acc_scores):.4f} (± {np.std(acc_scores):.4f})")


--- Outer Fold 1 / 5 ---
----------------------------------------
Best Params: {'svm__kernel': 'rbf', 'svm__C': 155.93732912711562, 'svm__gamma': 0.11850217009770807}
Best CV F1:  0.8373

--- Outer Fold 2 / 5 ---
----------------------------------------
Best Params: {'svm__kernel': 'rbf', 'svm__C': 217.65321491930786, 'svm__gamma': 0.1507234100145715}
Best CV F1:  0.8292

--- Outer Fold 3 / 5 ---
----------------------------------------
Best Params: {'svm__kernel': 'rbf', 'svm__C': 99.15316839362252, 'svm__gamma': 0.03856890578934132}
Best CV F1:  0.8363

--- Outer Fold 4 / 5 ---
----------------------------------------
Best Params: {'svm__kernel': 'rbf', 'svm__C': 226.93821615304316, 'svm__gamma': 0.07749752746046369}
Best CV F1:  0.8333

--- Outer Fold 5 / 5 ---
----------------------------------------
Best Params: {'svm__kernel': 'rbf', 'svm__C': 34.36195663601023, 'svm__gamma': 0.07934603879765877}
Best CV F1:  0.8337

---------- FINAL RESULTS ----------
최종 5-Fold 평균 F1 Score: 0.8

In [8]:
# 5. 최종 모델 학습 및 저장
import joblib

print(" [최종 모델 학습 시작]")
print("=========================================")

# 전체 데이터 사용하여 최종 최적화 수행
final_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
final_study.optimize(lambda trial: objective(trial, X, y, pipeline), n_trials=100, n_jobs=1)

final_params = final_study.best_params
final_score = final_study.best_value
    
print("-" * 40)
print(f"Final Hyper Params: {final_params}")
print(f"Final 5-cv F1:  {final_score:.4f}")
    
final_model = pipeline.set_params(**final_params)
final_model.fit(X, y) 

# 학습이 완료된 final_model 객체를 하드디스크에 파일로 저장합니다.
joblib.dump(final_model, os.path.join('..', 'vr_legibility_svm_model.pkl'))
print("✅ 최종 모델이 'vr_legibility_svm_model.pkl' 파일로 저장되었습니다!")

 [최종 모델 학습 시작]
----------------------------------------
Final Hyper Params: {'svm__kernel': 'rbf', 'svm__C': 18.84823984094989, 'svm__gamma': 0.1776213914832144}
Final 5-cv F1:  0.8361
✅ 최종 모델이 'vr_legibility_svm_model.pkl' 파일로 저장되었습니다!


In [ ]:
# 6. 모델 배포 후 예측 (새로운 데이터에 대한 예측)
import joblib
import pandas as pd

# 1. 저장해둔 모델 불러오기 
deployed_model = joblib.load(os.path.join('..', 'vr_legibility_svm_model.pkl'))

# 2. 새 데이터 불러오기
"""
new_data = pd.DataFrame({
    'size': [180.0, 160.0],
    'yaw': [1250.5, -450.2],
    'pitch': [45.2, 210.8],
    "label" :[1, 1]
})
"""
new_data = pd.read_csv(os.path.join('..', 'vr_legibility_train.csv'))[["yaw", "pitch", "size", "label"]]

X_new = new_data[["yaw", "pitch", "size"]]
y_true = new_data["label"]

# 3. 새로운 데이터 예측
predictions = deployed_model.predict(X_new)
print("새로운 데이터 예측 완료! (총 {}개)".format(len(predictions)))

acc = accuracy_score(y_true, predictions)
f1 = f1_score(y_true, predictions)
print("\n" + "-" * 40)
print(f"새로운 데이터에 대한 Accuracy: {acc:.4f}")
print(f"새로운 데이터에 대한 F1 Score: {f1:.4f}")

새로운 데이터 예측 완료! (총 1395개)

----------------------------------------
새로운 데이터에 대한 Accuracy: 0.8717
새로운 데이터에 대한 F1 Score: 0.8587
